# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdarshIsaac/NewRepoML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This is a **ranking / scoring** task. A reviewer has limited time, so the output should order content pages from highest to lowest review priority. The score will be based on signals available before review; it is decision support, not an automatic publishing decision.

In [1]:
from pathlib import Path
import pandas as pd


def find_dataset() -> Path:
    candidates = [
        Path.cwd() / "data" / "raw" / "content_refresh_anonymized.csv",
        Path.cwd().parent / "data" / "raw" / "content_refresh_anonymized.csv",
        Path.cwd().parent.parent / "data" / "raw" / "content_refresh_anonymized.csv",
        Path("data/raw/content_refresh_anonymized.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find the starter dataset.")


path = find_dataset()
df = pd.read_csv(path)

assert df["content_id"].is_unique, "The ranking unit must be one row per content item."
print("Dataset:", path)
print("Rows:", len(df))
print("Unique content items:", df["content_id"].nunique())
print("Ranking unit verified: one row = one content page")

Dataset: f:\GitHub\NewRepoML\data\raw\content_refresh_anonymized.csv
Rows: 30000
Unique content items: 30000
Ranking unit verified: one row = one content page


## 2. Target or proxy

The observed outcome is whether a page is declining: `is_declining_label = 1` when `trend_direction == "down"`. This label comes from measured search performance in the most recent 30-day window compared with the preceding 30-day window. `trend_direction` and `trend_pct` are label-source fields, so they are excluded from features to prevent leakage.

In [2]:
declining_labels = df["trend_direction"].fillna("").str.lower().eq("down").astype(int)

assert declining_labels.notna().all()
assert declining_labels.sum() > 0, "The observed outcome must contain positive examples."

print("Observed declining pages:", int(declining_labels.sum()))
print("Observed non-declining pages:", int((1 - declining_labels).sum()))
print("Label prevalence:", round(declining_labels.mean() * 100, 1), "%")
print("Leakage fields excluded from features:", ["trend_direction", "trend_pct"])
print("Identifier fields excluded from features:", ["content_id", "client_id"])

Observed declining pages: 16262
Observed non-declining pages: 13738
Label prevalence: 54.2 %
Leakage fields excluded from features: ['trend_direction', 'trend_pct']
Identifier fields excluded from features: ['content_id', 'client_id']


## 3. Success metric

The primary metric is **Precision@50**: among the 50 pages ranked highest for review, the share that are observed declining pages. This matches the decision because a reviewer has a small fixed queue. A useful model must beat the prevalence baseline, which is the expected precision of selecting pages at random; the exact score will be measured after a leakage-safe model is trained.

In [3]:
k = 50
random_precision_at_k = declining_labels.mean()

assert len(df) >= k, "The review queue must contain at least K pages."
print("Metric:", f"Precision@{k}")
print("Definition: declining pages in the top", k, "/", k)
print("Random-selection baseline Precision@50:", round(random_precision_at_k, 3))
print("A ranking is useful only if its measured Precision@50 exceeds this baseline on held-out data.")

Metric: Precision@50
Definition: declining pages in the top 50 / 50
Random-selection baseline Precision@50: 0.542
A ranking is useful only if its measured Precision@50 exceeds this baseline on held-out data.


## 4. The unit of analysis, as a real dataframe

One row represents one pseudonymized content page. The starter slice contains page-level features and aggregated trailing-90-day search and analytics signals, with a decline outcome derived from the comparison of two 30-day periods. The page ID identifies the row but is used only for reporting and grouping, never as a predictive feature.

In [4]:
display_columns = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "trend_direction",
]
page_view = df[display_columns].copy()
page_view["is_declining_label"] = declining_labels

assert page_view["content_id"].is_unique
assert len(page_view) == len(df)
print("Dataframe shape:", page_view.shape)
print("One row = one content page:", page_view["content_id"].nunique() == len(page_view))
display(page_view.head(5))

Dataframe shape: (30000, 9)
One row = one content page: True


,content_id,client_id,content_type,impressions_90d,sessions_90d,ctr,avg_position,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,17,0.76,10.6,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,9,0.05,20.3,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,78,0.49,6.2,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,145,0.13,44.0,down,1


## 5. Why ML beats a fixed rule here

A single threshold, such as “review pages below position 20,” will miss interactions between demand, traffic, engagement, content properties, and freshness. A learned score can combine these signals and rank a large queue consistently. The comparison must remain honest: ML earns its place only if it improves held-out `Precision@50` over a transparent fixed-rule or random baseline.

In [5]:
candidate_signal_columns = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "content_type",
]
available_signals = [column for column in candidate_signal_columns if column in df.columns]

assert len(available_signals) >= 5, "The lane needs several available signals to justify a learned ranking."
assert df[available_signals].nunique(dropna=False).gt(1).all(), "Each candidate signal should vary."

print("Available candidate signals:", len(available_signals))
print("Signals:", available_signals)
print("Why a learned score is testable: these signals vary across pages and can be evaluated against held-out observed decline outcomes.")

Available candidate signals: 9
Signals: ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'content_age_days', 'days_since_last_update', 'content_type']
Why a learned score is testable: these signals vary across pages and can be evaluated against held-out observed decline outcomes.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.